# Khám phá dữ liệu vận hành của 14 doanh nghiệp

Nguồn phân tích là bộ DataMate `public_test`. Bắt đầu từ các câu hỏi về độ phủ và chất lượng của 14 doanh nghiệp, sau đó tìm hiểu phân phối giá trị đơn ở dev_001 và đối chiếu nhịp bán với dev_009.

**Cách kiểm tra số liệu.** Xác định một dòng đại diện cho điều gì, đọc README và rulebook, kiểm tra ngày và trạng thái, rồi mới gộp dòng về mã đơn. Kết luận chỉ áp dụng cho các nguồn và kỳ được ghi trên biểu đồ.


In [ ]:
import sys
from pathlib import Path

SCRIPT_ROOT = Path.cwd() / 'src' / 'eda' / 'script'
if not SCRIPT_ROOT.exists():
    SCRIPT_ROOT = Path.cwd().resolve()
    for candidate in SCRIPT_ROOT.parents:
        if (candidate / 'src' / 'eda' / 'script').is_dir():
            SCRIPT_ROOT = candidate / 'src' / 'eda' / 'script'
            break
if str(SCRIPT_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPT_ROOT))

from IPython.display import HTML, Image, Markdown, display

from common.config import load_config
from orchestration.notebook_builder import course_figure_limits
from orchestration.pipeline import run_eda
from orchestration.study_examples import (
    load_order_studies, order_audit, order_value_figure, weekly_activity_figure,
)

config = load_config()
run = run_eda(config, figure_limits=course_figure_limits())
run.run_id


In [ ]:
def _show_figure(chart_id, split='public_test', limit=1):
    html = run.figures_html(chart_id, split, limit)
    if html:
        display(HTML(html))


In [ ]:
def _show_table(table_id, split=None):
    html = run.table_html(table_id, split)
    if not html:
        print(f'Bảng {table_id} không có dữ liệu cho split {split}')
        return
    display(HTML(html))


## 00. Cấu hình, chạy pipeline và provenance

**Mục tiêu:** Xác lập cấu hình, chạy pipeline EDA một lần và hiển thị ledger kiểm định cùng provenance của toàn bộ artifact sinh ra.

**Phương pháp:** Pipeline đọc nguồn ở chế độ chỉ đọc, tính toàn bộ chỉ số tại thời điểm nạp dữ liệu và ghi registry có hash cho từng artifact.

**Cách đọc và hạn chế:** Mọi con số trong notebook đều truy vết được về run_id, source hash và phương pháp tính; không có số liệu nào nhập tay.


In [ ]:
_show_table('TBL20')


## 01. Tổng quan bộ dữ liệu

**Mục tiêu:** Mô tả phạm vi dữ liệu theo split, đơn vị đếm và mức độ bao phủ của từng doanh nghiệp.

**Phương pháp:** Đếm đối tượng siêu dữ liệu từ manifest và registry của từng split; số tệp và số giao dịch là hai đại lượng khác nhau.

**Cách đọc và hạn chế:** Tổng số tệp, số worksheet và số đối tượng CSDL là số đối tượng nguồn, không phải số giao dịch và không được cộng chéo.


In [ ]:
_show_table('TBL01')

_show_figure('INV01', 'public_test', 1)
_show_figure('GLOBAL06', 'public_test', 1)


## 02. Kiểm kê nguồn dữ liệu

**Mục tiêu:** Liệt kê toàn bộ tệp, worksheet và đối tượng CSDL kèm trạng thái đọc, hash và siêu dữ liệu cấu trúc.

**Phương pháp:** Registry kiểm kê gom từ manifest nguồn và registry worksheet/database; mỗi bản ghi gắn content hash của tệp gốc.

**Cách đọc và hạn chế:** Nhiều tệp hơn không đồng nghĩa nhiều giao dịch hơn; tệp trong manifest được đọc ở chế độ chỉ đọc và không bị thay đổi.


In [ ]:
_show_table('TBL02', 'public_test')
_show_table('TBL03', 'public_test')
_show_table('TBL04', 'public_test')

_show_figure('INV01', 'public_test', 1)
_show_figure('INV02', 'public_test', 1)
_show_figure('INV03', 'public_test', 1)
_show_figure('INV04', 'public_test', 1)


## 03. Khám phá schema và mối quan hệ

**Mục tiêu:** Xác định kiểu vật lý và ngữ nghĩa của từng cột, cùng bằng chứng khóa chính và quan hệ giữa các bảng.

**Phương pháp:** Suy luận kiểu dữ liệu trên toàn bộ dòng đã đọc, kết hợp bằng chứng khóa ứng viên và kiểm tra số khóa lệch.

**Cách đọc và hạn chế:** Quan hệ ứng viên chưa được phê duyệt không được dùng để tính chỉ số; phép nối có rủi ro nhân bản bị chặn theo thiết kế.


In [ ]:
_show_table('TBL05', 'public_test')
_show_table('TBL12', 'public_test')

_show_figure('GLOBAL02', 'public_test', 1)
_show_figure('QUAL07', 'public_test', 1)


## 04. Chất lượng dữ liệu

**Mục tiêu:** Đo mức thiếu dữ liệu, trùng lặp, vi phạm quy tắc và mức độ phát hiện theo từng nguồn, kèm mẫu số rõ ràng.

**Phương pháp:** Mọi tỷ lệ đều chia cho mẫu số của chính trường hoặc chính quy tắc được hồ sơ; thiếu dữ liệu có cấu trúc tách riêng khỏi khoảng trống thực.

**Cách đọc và hạn chế:** Cờ thiếu dữ liệu và cờ trùng lặp là tín hiệu thống kê, chưa được xác định là bản ghi lỗi nếu chưa có đối chiếu rulebook.


In [ ]:
_show_table('TBL06', 'public_test')
_show_table('TBL07', 'public_test')
_show_table('TBL08', 'public_test')
_show_table('TBL19', 'public_test')

_show_figure('QUAL01', 'public_test', 1)
_show_figure('QUAL02', 'public_test', 1)
_show_figure('QUAL04', 'public_test', 1)
_show_figure('QUAL05', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 05. Phân tích định lượng

**Mục tiêu:** Mô tả phân phối các đo lường định lượng trên toàn bộ dân số hợp lệ, kèm thống kê bền vững và cờ ngoại lệ theo IQR.

**Phương pháp:** Tính quantile, MAD, lệch và kurtosis trên toàn bộ quan sát hợp lệ; ngưỡng ngoại lệ IQR chỉ gắn cờ, không tự động loại bỏ.

**Cách đọc và hạn chế:** Kiểu vật lý không bảo đảm ngữ nghĩa đo lường; cột định danh dạng số được xử lý riêng và không đưa vào phân tích phân phối đo lường.


In [ ]:
_show_table('TBL09', 'public_test')

_show_figure('NUM01', 'public_test', 2)
_show_figure('NUM02', 'public_test', 1)
_show_figure('NUM06', 'public_test', 1)
_show_figure('NUM08', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 06. Phân tích phân loại

**Mục tiêu:** Khảo sát phân phối tần suất của các trường phân loại, giữ nguyên nhóm Others và ghi nhận ứng viên bí danh thay vì tự gộp.

**Phương pháp:** Đếm tần suất theo giá trị đã chuẩn hóa, sắp xếp theo tần suất giảm dần và ghi rõ phạm vi quan sát của mỗi trường.

**Cách đọc và hạn chế:** Cao độ lớp phụ thuộc quy tắc chuẩn hóa giá trị; các giá trị tương tự nhau về nghĩa chỉ được báo cáo là ứng viên, không tự ý hợp nhất.


In [ ]:
_show_table('TBL10', 'public_test')

_show_figure('CAT01', 'public_test', 2)
_show_figure('CAT02', 'public_test', 1)
_show_figure('CAT03', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 07. Tương quan và phân tích đa biến

**Mục tiêu:** Đo mức liên hệ giữa các đo lường và các trường phân loại trên cùng grain, chỉ dùng các phép nối đã được phê duyệt.

**Phương pháp:** Tính Pearson, Spearman, Kendall, Cramér V và bảng chéo trong phạm vi cùng một nguồn hoặc phép nối không nhân bản.

**Cách đọc và hạn chế:** Tương quan không phải quan hệ nhân quả; phép nối ứng viên chỉ được hiển thị để đánh giá rủi ro nhân bản, không dùng cho tính chỉ số.


In [ ]:
_show_table('TBL12', 'public_test')

_show_figure('REL01', 'public_test', 1)
_show_figure('REL03', 'public_test', 1)
_show_figure('REL06', 'public_test', 1)
_show_figure('REL07', 'public_test', 1)
_show_figure('REL08', 'public_test', 1)
_show_figure('REL09', 'public_test', 1)
_show_figure('REL10', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 08. Hồ sơ thời gian và lịch

**Mục tiêu:** Xác định ngữ nghĩa trường ngày, khoảng phủ thời gian, số ngày có hoạt động và các kỳ bị thiếu theo lịch của split.

**Phương pháp:** Đọc và chuẩn hóa giá trị ngày theo quy tắc từng split, đối chiếu với lịch và báo cáo khoảng phủ theo doanh nghiệp.

**Cách đọc và hạn chế:** Mỗi trường ngày giữ vai trò sự kiện riêng nên các khoảng phủ không thể so sánh trực tiếp; thiếu dữ liệu theo kỳ không được coi là giá trị bằng không.


In [ ]:
_show_table('TBL11', 'public_test')

_show_figure('TIME01', 'public_test', 1)
_show_figure('TIME02', 'public_test', 1)
_show_figure('TIME03', 'public_test', 1)
_show_figure('TIME07', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 09. Phân tích theo từng doanh nghiệp

**Mục tiêu:** Trình bày hồ sơ, phân loại ngành và các chỉ số kinh doanh đủ điều kiện của từng doanh nghiệp trong split.

**Phương pháp:** Chỉ số được tính trên grain đã xác định, tuân thủ quy tắc trạng thái của rulebook split và chỉ hiển thị khi doanh nghiệp đủ điều kiện.

**Cách đọc và hạn chế:** Doanh nghiệp thiếu điều kiện bị ghi trạng thái rõ ràng, không được thay bằng số liệu của doanh nghiệp khác để lấp chỗ trống.


In [ ]:
_show_table('TBL13', 'public_test')
_show_table('TBL14', 'public_test')


## 10. Giá trị đơn hàng và ngoại lệ tại dev_001

**Mục tiêu:** Giá trị tiền hàng điển hình của một đơn là bao nhiêu? Các đơn lớn nằm ở đâu trong phân phối?

**Phương pháp:** Tại dev_001, khóa là mã đơn và doanh thu là tổng Thành tiền các dòng cùng đơn. Tại dev_009, khóa là mã hóa đơn và doanh thu lấy từ Giá trị hàng. Bảng kiểm tra ghi số dòng thiếu mã, dòng trùng y hệt và số đơn thiếu dữ liệu hoặc mâu thuẫn. Giữ đơn hoàn thành và đang giao khi tính doanh thu. Xác định ngoại lệ ở cấp đơn bằng hai ngưỡng Q1 − 1,5 × IQR và Q3 + 1,5 × IQR, với IQR = Q3 − Q1.

**Cách đọc và hạn chế:** Dòng thiếu mã được loại khỏi phân tích đơn vì chưa thể xác minh thuộc đơn nào. Cờ IQR dùng để kiểm tra lại chứng từ. Các đơn ngoài ngưỡng vẫn góp vào thống kê.


In [ ]:
studies = load_order_studies(config.split_root('public_test'))
case_figures = run.output_root / 'figures' / 'course'
display(order_audit(studies))
_, iqr_summary = order_value_figure(studies[0], case_figures)
display(iqr_summary.round(2))
display(Image(filename=str(case_figures / 'phan-phoi-gia-tri-don.png'), width=960))


## 11. Đối chiếu nhịp bán giữa dev_001 và dev_009

**Mục tiêu:** Số đơn hợp lệ trung bình trên từng thứ trong tuần thay đổi thế nào giữa hai doanh nghiệp?

**Phương pháp:** Chọn hai nguồn có mã đơn, ngày và trạng thái đủ để xác định đơn hợp lệ, với hai cách ghi khác nhau. Giữ các tháng mà mỗi ngày đều có ít nhất một đơn ghi nhận được ở cả hai nguồn. Chia số đơn hợp lệ theo thứ cho số ngày lịch của thứ đó trong các tháng chung.

**Cách đọc và hạn chế:** Mỗi cột luôn bắt đầu từ 0 và hai ô biểu đồ dùng chung thang đo. Kết quả mô tả hai doanh nghiệp trong các tháng ghi dưới bảng.


In [ ]:
weekly_plot, weekly_summary = weekly_activity_figure(studies, case_figures)
if weekly_plot is None:
    display(Markdown('Hai doanh nghiệp chưa có tháng nào cùng đủ ngày quan sát.'))
else:
    display(Markdown('Các tháng đối chiếu: ' + weekly_summary.attrs['months']))
    display(weekly_summary.round(2))
    display(Image(filename=str(case_figures / 'nhip-ban-theo-thu.png'), width=960))


## 12. Quy tắc nghiệp vụ theo Rulebook Public Test

**Mục tiêu:** Tổng hợp các quy tắc định nghĩa chỉ số và chuẩn nghiệp vụ trong rulebook.md của Public Test.

**Phương pháp:** Trích xuất quy tắc theo bảng và mục từ rulebook.md của Public Test.

**Cách đọc và hạn chế:** Quy tắc của Public Test là chuẩn duy nhất áp dụng cho toàn bộ phân tích nghiệp vụ.


In [ ]:
_show_table('TBL16')


## 13. So sánh doanh nghiệp trên toàn hệ thống

**Mục tiêu:** Đối chiếu sự hiện diện của vai trò trường ngữ nghĩa giữa các doanh nghiệp và khả năng dựng biểu đồ theo nhóm.

**Phương pháp:** Phát hiện vai trò trường theo tên và đối chiếu trạng thái đủ điều kiện của từng nhóm biểu đồ theo doanh nghiệp.

**Cách đọc và hạn chế:** Cùng một vai trò ngữ nghĩa không đồng nghĩa với khóa nối tương đương; đủ điều kiện về schema không bảo đảm biểu đồ dựng được.


In [ ]:
_show_table('TBL17', 'public_test')

_show_figure('GLOBAL02', 'public_test', 1)
_show_figure('GLOBAL03', 'public_test', 1)
_show_figure('GLOBAL04', 'public_test', 1)
_show_figure('GLOBAL05', None, 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:2600]))


## 14. Phân tích dữ liệu toàn cục

**Mục tiêu:** Kiểm tra mức độ bao phủ của 14 doanh nghiệp ở mức siêu dữ liệu.

**Phương pháp:** Tổng hợp số tệp, số doanh nghiệp, số worksheet, số đối tượng CSDL và số phát hiện chất lượng theo từng split.

**Cách đọc và hạn chế:** Số lượng đối tượng siêu dữ liệu không phải số giao dịch và không được cộng chéo giữa các loại đối tượng.


In [ ]:
_show_table('TBL18')

_show_figure('GLOBAL01', 'public_test', 1)
_show_figure('GLOBAL06', 'public_test', 1)


## 15. Chất lượng và mức độ sẵn sàng cho tác nhân

**Mục tiêu:** Trình bày ledger kiểm định theo phase, trạng thái phát hiện chất lượng và danh mục phát hiện có bằng chứng.

**Phương pháp:** Mỗi gate ghi rõ phase, trạng thái, lý do và artifact chứng minh; mỗi phát hiện gắn mức độ, tỷ lệ và bằng chứng nguồn.

**Cách đọc và hạn chế:** Phân biệt phát hiện có bằng chứng với lỗi đã xác nhận; trạng thái BLOCKED là kết quả hợp lệ và không được nâng lên PASS để đạt hình thức.


In [ ]:
_show_table('TBL19', 'public_test')
_show_table('TBL20')

_show_figure('QUAL06', 'public_test', 1)
_show_figure('QUAL08', None, 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:2600]))


## 16. Kết luận khoa học và bàn giao

**Mục tiêu:** Tổng hợp kết luận khoa học, hạn chế đã xác định và dòng dẫn tới từng artifact có hash để tái lập.

**Phương pháp:** Mỗi kết luận gắn mức tin cậy và đường dẫn bằng chứng; thẻ chuyện khoa học đính kèm theo từng nhóm nghiên cứu.

**Cách đọc và hạn chế:** Không có kết luận nào vượt quá phạm vi bằng chứng đã ghi; phần chưa đủ bằng chứng được nêu rõ thay vì bị lược bỏ.


In [ ]:
_show_figure('GLOBAL07', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:15600]))
